In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch
import numpy as np
import src.tools as tools

from importlib import reload

path_to_file = keras.utils.get_file(
        'shakespeare.txt',
        'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
        )
print(path_to_file)



## Load and preprocess Shakespeare text

In [67]:
with open(path_to_file) as f:
    shakespeare_text = f.read()
print(shakespeare_text[:500])

characters = sorted(set(shakespeare_text))
char_to_id = {char: index for index, char in enumerate(characters)}

shakespeare_tensor = torch.tensor(
        [char_to_id[char] for char in shakespeare_text],
        dtype=torch.long,
        )

print(shakespeare_tensor.shape)  # (text_length,)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
torch.Size([1115394])


In [68]:
vocab = list(set(shakespeare_text.lower().strip()))
print(vocab)
print(f"vocab size = {len(vocab)}")

[':', 'x', '&', '?', 'w', 'p', 'o', 'f', '!', 'a', '$', 'r', 'b', 'v', 'q', '-', 'd', ';', 'l', 'e', 'g', 'y', 'n', "'", '3', 's', ' ', 'm', 'k', 'u', 't', '\n', 'i', 'c', 'h', '.', 'j', 'z', ',']
vocab size = 39


In [69]:
tokenizer = tools.CharacterTokenizer(vocab)
encoded_text = tokenizer.encode(shakespeare_text)
print(f"Encoded text: {encoded_text[:100]}")

Encoded text: tensor([19, 22, 31, 32, 33,  2, 16, 22, 33, 22, 39, 18, 27, 11,  1, 15, 18, 19,
        28, 31, 18,  2, 36, 18,  2, 29, 31, 28, 16, 18, 18, 17,  2, 14, 27, 38,
         2, 19, 34, 31, 33, 21, 18, 31,  7,  2, 21, 18, 14, 31,  2, 26, 18,  2,
        32, 29, 18, 14, 24,  9,  1,  1, 14, 25, 25, 11,  1, 32, 29, 18, 14, 24,
         7,  2, 32, 29, 18, 14, 24,  9,  1,  1, 19, 22, 31, 32, 33,  2, 16, 22,
        33, 22, 39, 18, 27, 11,  1, 38, 28, 34])


### One-hot encoded Dataset

In [70]:
reload(tools)

n_steps = 50
batch_size = 64
train_dataset_oh = tools.text_to_dataset(
        encoded, n_tokens, n_steps=n_steps, batch_size=batch_size
        )

X, y = next(iter(train_dataset_oh))
print(f"X = {X.shape}")  # (64, 50, n_tokens)
print(f"y = {y.shape}")  # (64, 50)

X = torch.Size([64, 50, 40])
y = torch.Size([64, 50])


### Baseline: RNN with GRU layer

In [73]:
m1 = keras.Sequential(
        [
                keras.layers.Input(shape=(None, n_tokens)),
                keras.layers.GRU(128, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax")
                ],
        )
m1.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
m1.summary()

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_16 (GRU)                    │ (None, None, 128)      │        65,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, None, 40)       │         5,160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 70,440 (275.16 KB)

 Trainable params: 70,440 (275.16 KB)

 Non-trainable params: 0 (0.00 B)

In [71]:
import torch.nn.functional as F

model = m1
text = "to be or no"
tokens = tokenizer.encode(text)
print(tokens)
input = F.one_hot(tokens, num_classes=n_tokens).float()
print(input[0])
print(input.shape)
input = input.unsqueeze(0)  # Add batch dimension
print(input.shape)

tensor([33, 28,  2, 15, 18,  2, 28, 31,  2, 27, 28])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
        0., 0., 0., 0.])
torch.Size([11, 40])
torch.Size([1, 11, 40])


In [25]:
prediction = model(input)
print(prediction.shape)

torch.Size([1, 11, 40])


In [28]:
print(prediction[0, :, :].argmax(axis=-1))
next_token = prediction[0, -1, :].argmax(axis=-1)
print(next_token)

tensor([22, 26, 26,  9,  9, 19, 23, 24, 24, 12, 26], device='mps:0')
tensor(26, device='mps:0')


In [29]:
tokenizer.decode(next_token)

'm'

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m1.fit(
        train_dataset_oh,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )

In [75]:
m1.save("../models/shakespeare_gru_1_oh.keras")

In [76]:
m1 = keras.models.load_model("../models/shakespeare_gru_1_oh.keras")
prediction = m1(input)
print(prediction.shape)
print(prediction[0, :, :].argmax(axis=-1))
next_token = prediction[0, -1, :].argmax(axis=-1)
tokenizer.decode(next_token)

torch.Size([1, 11, 40])
tensor([21,  2, 33, 18,  2, 32, 19, 17, 14, 28, 33], device='mps:0')


't'

In [78]:
tools.generate_text(m1, tokenizer, "to be", n_next=100, one_hot=True)

'to be so see the service.\n\nantonio:\nwhat is not see the world is not see the world\nand see the service, s'

In [81]:
m2 = keras.Sequential(
        [
                keras.layers.Input(shape=(None, n_tokens)),
                keras.layers.GRU(128, return_sequences=True),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax")
                ],
        )
m2.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
m2.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_19 (GRU)                    │ (None, None, 128)      │        65,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_20 (GRU)                    │ (None, None, 64)       │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, None, 40)       │         2,600 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,128 (410.66 KB)

 Trainable params: 105,128 (410.66 KB)

 Non-trainable params: 0 (0.00 B)

In [82]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m2.fit(
        train_dataset_oh,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m2.save("../models/shakespeare_gru_2_oh.keras")
tools.generate_text(m2, tokenizer, "to be", n_next=100, one_hot=True)

Epoch 1/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 776s 45ms/step - loss: 1.7253 
Epoch 2/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 762s 44ms/step - loss: 1.5152 
Epoch 3/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 750s 43ms/step - loss: 1.4627 
Epoch 4/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 749s 43ms/step - loss: 1.4359 
Epoch 5/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 745s 43ms/step - loss: 1.4190 
Epoch 6/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 750s 43ms/step - loss: 1.4076 
Epoch 7/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 747s 43ms/step - loss: 1.3991 


'to be so soundly seem the sea\nwith the sea with the sea with the season and soundly seem the sea\nwith the'

### Deep RNN with 3 GRU layers

In [83]:
m3 = keras.Sequential(
        [
                keras.layers.Input(shape=(None, n_tokens)),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax")
                ],
        )
m3.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
m3.summary()

Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_21 (GRU)                    │ (None, None, 64)       │        20,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_22 (GRU)                    │ (None, None, 64)       │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_23 (GRU)                    │ (None, None, 64)       │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, None, 40)       │         2,600 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 72,872 (284.66 KB)

 Trainable params: 72,872 (284.66 KB)

 Non-trainable params: 0 (0.00 B)

In [84]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m3.fit(
        train_dataset_oh,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m3.save("../models/shakespeare_gru_3_oh.keras")
tools.generate_text(m3, tokenizer, "to be", n_next=100, one_hot=True)

Epoch 1/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 1077s 62ms/step - loss: 1.7734
Epoch 2/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 1191s 68ms/step - loss: 1.5664
Epoch 3/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 1250s 72ms/step - loss: 1.5166
Epoch 4/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 1246s 72ms/step - loss: 1.4911
Epoch 5/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 1291s 74ms/step - loss: 1.4753
Epoch 6/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 1192s 68ms/step - loss: 1.4649
Epoch 7/10
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 1125s 65ms/step - loss: 1.4571


'to be so stay the sea and so so so so so so so so so so so so so so so so so so so so so so so so so so s'

## One-hot layer as a custom Keras layer

In [91]:
class OneHotLayer(keras.layers.Layer):
    def __init__(self, n_tokens, **kwargs):
        super().__init__(**kwargs)
        self.n_tokens = n_tokens

    def call(self, inputs):
        return keras.ops.one_hot(inputs, num_classes=self.n_tokens)

    def compute_output_shape(self, input_shape):
        return (*input_shape, self.n_tokens)


m3 = keras.Sequential(
        [
                keras.layers.Input(shape=(None,), dtype="int32"),
                OneHotLayer(n_tokens),
                keras.layers.GRU(128, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax"),
                ]
        )

m3.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam"
        )

m3.summary()

Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ one_hot_layer_6 (OneHotLayer)   │ (None, None, 40)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_25 (GRU)                    │ (None, None, 128)      │        65,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, None, 40)       │         5,160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 70,440 (275.16 KB)

 Trainable params: 70,440 (275.16 KB)

 Non-trainable params: 0 (0.00 B)

In [88]:
text = "to be or no"
tokens = tokenizer.encode(text)
print(tokens)
input = torch.tensor(tokens, dtype=torch.long)
print(input.shape)
input = input.unsqueeze(0)  # Add batch dimension
print(input.shape)
prediction = m3(input)
print(prediction.shape)
print(prediction[0, :, :].argmax(axis=-1))
next_token = prediction[0, -1, :].argmax(axis=-1)
tokenizer.decode(next_token)

tensor([33, 28,  2, 15, 18,  2, 28, 31,  2, 27, 28])
torch.Size([11])
torch.Size([1, 11])
torch.Size([1, 11, 40])
tensor([27, 28, 27, 11,  1,  2, 28, 28,  8, 20, 28], device='mps:0')


/var/folders/fn/631f5l8d60s9dlhzgthzclk80000gn/T/ipykernel_12958/746064773.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input = torch.tensor(tokens, dtype=torch.long)


'o'

In [89]:
reload(tools)

n_steps = 50
batch_size = 128
train_dataset = tools.text_to_dataset(
        encoded, n_steps=n_steps, batch_size=batch_size
        )

X, y = next(iter(train_dataset))
print(f"X = {X.shape}")  # (64, 50)
print(f"y = {y.shape}")  # (64, 50)

X = torch.Size([128, 50])
y = torch.Size([128, 50])


In [92]:
keras.config.disable_traceback_filtering()

torch._dynamo.config.verbose = True

early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m3.fit(
        train_dataset,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m3.save("../models/shakespeare_gru_3.keras")
tools.generate_text(m3, tokenizer, "to be", n_next=100, one_hot=False)

Epoch 1/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 341s 39ms/step - loss: 1.8795
Epoch 2/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 266s 31ms/step - loss: 1.6080
Epoch 3/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 268s 31ms/step - loss: 1.5353
Epoch 4/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 269s 31ms/step - loss: 1.4979
Epoch 5/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 268s 31ms/step - loss: 1.4741
Epoch 6/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 268s 31ms/step - loss: 1.4579
Epoch 7/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 267s 31ms/step - loss: 1.4460
Epoch 8/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 265s 30ms/step - loss: 1.4363


'to be son of the service of the service.\n\nantonio:\nthe world the matter of the service of the service.\n\na'

## Embedding Layer

In [95]:
m5 = keras.Sequential(
        [
                keras.layers.Input(shape=(None,), dtype="int32"),
                keras.layers.Embedding(input_dim=n_tokens, output_dim=8),
                keras.layers.GRU(128, return_sequences=True),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax"),
                ]
        )
m5.compile(loss="sparse_categorical_crossentropy", optimizer="nadam")
m5.summary()

Model: "sequential_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, None, 8)        │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_28 (GRU)                    │ (None, None, 128)      │        52,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_29 (GRU)                    │ (None, None, 64)       │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, None, 40)       │         2,600 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,160 (363.91 KB)

 Trainable params: 93,160 (363.91 KB)

 Non-trainable params: 0 (0.00 B)

In [96]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m5.fit(
        train_dataset,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m5.save("../models/shakespeare_gru_5.keras")
tools.generate_text(m5, tokenizer, "to be", n_next=100, one_hot=False)

Epoch 1/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 523s 60ms/step - loss: 1.9218
Epoch 2/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 503s 58ms/step - loss: 1.6273
Epoch 3/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 477s 55ms/step - loss: 1.5491
Epoch 4/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 482s 55ms/step - loss: 1.5089
Epoch 5/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 476s 55ms/step - loss: 1.4849
Epoch 6/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 458s 53ms/step - loss: 1.4687
Epoch 7/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 444s 51ms/step - loss: 1.4570
Epoch 8/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 444s 51ms/step - loss: 1.4474


'to be so should not the sun of the sense.\n\ngonzalo:\nand the senses and son the sense,\nwho is the sense, w'

## LSTM Layer

In [98]:
m6 = keras.Sequential(
        [
                keras.layers.Input(shape=(None,), dtype="int32"),
                keras.layers.Embedding(input_dim=n_tokens, output_dim=8),
                keras.layers.LSTM(64, return_sequences=True),
                keras.layers.LSTM(64, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax"),
                ]
        )
m6.compile(loss="sparse_categorical_crossentropy", optimizer="nadam")
m6.summary()

Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, None, 8)        │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, None, 64)       │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, None, 64)       │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, None, 40)       │         2,600 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,632 (213.41 KB)

 Trainable params: 54,632 (213.41 KB)

 Non-trainable params: 0 (0.00 B)

In [99]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m6.fit(
        train_dataset,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m6.save("../models/shakespeare_lstm_6.keras")
tools.generate_text(m6, tokenizer, "to be", n_next=100, one_hot=False)

Epoch 1/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 435s 50ms/step - loss: 2.2579
Epoch 2/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 435s 50ms/step - loss: 1.8504
Epoch 3/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 423s 49ms/step - loss: 1.7134
Epoch 4/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 422s 48ms/step - loss: 1.6428
Epoch 5/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 414s 48ms/step - loss: 1.6004
Epoch 6/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 412s 47ms/step - loss: 1.5712
Epoch 7/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 407s 47ms/step - loss: 1.5495
Epoch 8/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 411s 47ms/step - loss: 1.5328
Epoch 9/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 493s 57ms/step - loss: 1.5194
Epoch 10/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 431s 50ms/step - loss: 1.5082


'to be the son.\n\nantonio:\nwhat i say the son the son the son.\n\nantonio:\nwhat i say the son the son the son'

## Byte Pair Encoding (BPE) Tokenizer

In [123]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# BPE-Tokenizer erstellen
bpe_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()

# Vokabular trainieren
trainer = trainers.BpeTrainer(
        vocab_size=100,
        special_tokens=["[UNK]"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
        )

bpe_tokenizer.train_from_iterator([shakespeare_text], trainer=trainer)
n_bpe_tokens = bpe_tokenizer.get_vocab_size()
print("BPE-Vokabulargröße:", n_bpe_tokens)
# Text tokenisieren
ids = bpe_tokenizer.encode(shakespeare_text).ids

print(f"Tokenanzahl: {len(ids):,}, Charakteranzahl: {len(shakespeare_text):,}")
print("Bei spiel:", ids[:30])

# Zurück in Text
print(bpe_tokenizer.decode(ids[:30]))




BPE-Vokabulargröße: 257
Tokenanzahl: 1,115,394, Charakteranzahl: 1,115,394
Bei spiel: [38, 73, 82, 83, 84, 221, 35, 73, 84, 73, 90, 69, 78, 26, 199, 34, 69, 70, 79, 82, 69, 221, 87, 69, 221, 80, 82, 79, 67, 69]
First Citizen:
Before we proce


In [124]:
bpe_ds = tools.text_to_dataset(ids, n_steps=50, batch_size=128)

In [127]:
m7 = keras.Sequential(
        [
                keras.layers.Input(shape=(None,), dtype="int32"),
                keras.layers.Embedding(input_dim=n_bpe_tokens, output_dim=64),
                keras.layers.GRU(128, return_sequences=True),
                # keras.layers.GRU(64, return_sequences=True),
                keras.layers.Dense(n_bpe_tokens, activation="softmax"),
                ]
        )
m7.compile(loss="sparse_categorical_crossentropy", optimizer="nadam")
m7.summary()

Model: "sequential_32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_13 (Embedding)        │ (None, None, 16)       │         4,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_38 (GRU)                    │ (None, None, 128)      │        56,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, None, 257)      │        33,153 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,329 (364.57 KB)

 Trainable params: 93,329 (364.57 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m7.fit(
        bpe_ds,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m7.save("../models/shakespeare_lstm_7.keras")
tools.generate_text(m7, tokenizer, "to be", n_next=100, one_hot=False)

In [136]:
reload(tools)
tools.generate_text(m7, bpe_tokenizer, "to be", n_next=100, one_hot=False)

'to be so lives the seem.\n\nANTONIO:\nWhat she speak the seem and so lives the seem.\n\nANTONIO:\nWhat she spea'

In [142]:
bpe_tokenizer_512 = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe_tokenizer_512.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer_512.decoder = decoders.ByteLevel()

# Vokabular trainieren
trainer = trainers.BpeTrainer(
        vocab_size=512,
        special_tokens=["[UNK]"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
        )

bpe_tokenizer_512.train_from_iterator([shakespeare_text], trainer=trainer)
n_bpe_tokens = bpe_tokenizer_512.get_vocab_size()
print("BPE-Vokabulargröße:", n_bpe_tokens)
# Text tokenisieren
ids = bpe_tokenizer_512.encode(shakespeare_text).ids

print(f"Tokenanzahl: {len(ids):,}, Charakteranzahl: {len(shakespeare_text):,}")
print("Beispiel:", ids[:30])

# Zurück in Text
print(bpe_tokenizer_512.decode(ids[:30]))




BPE-Vokabulargröße: 512
Tokenanzahl: 575,809, Charakteranzahl: 1,115,394
Beispiel: [38, 314, 296, 421, 275, 73, 90, 280, 26, 199, 34, 69, 70, 370, 332, 290, 371, 309, 316, 404, 89, 272, 362, 84, 336, 12, 293, 285, 318, 411]
First Citizen:
Before we proceed any further, hear me sp


In [147]:
bpe_tokenizer_512.save("../models/bpe_tokenizer_512.json")

In [145]:
m8 = keras.Sequential(
        [
                keras.layers.Input(shape=(None,), dtype="int32"),
                keras.layers.Embedding(input_dim=n_bpe_tokens, output_dim=64),
                keras.layers.GRU(128, return_sequences=True),
                # keras.layers.GRU(64, return_sequences=True),
                keras.layers.Dense(n_bpe_tokens, activation="softmax"),
                ]
        )
m8.compile(loss="sparse_categorical_crossentropy", optimizer="nadam")
m8.summary()

Model: "sequential_34"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_15 (Embedding)        │ (None, None, 64)       │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_40 (GRU)                    │ (None, None, 128)      │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, None, 512)      │        66,048 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 173,312 (677.00 KB)

 Trainable params: 173,312 (677.00 KB)

 Non-trainable params: 0 (0.00 B)

In [146]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m8.fit(
        bpe_ds,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m8.save("../models/shakespeare_lstm_8.keras")
tools.generate_text(m8, bpe_tokenizer, "to be", n_next=100, one_hot=False)

Epoch 1/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 319s 37ms/step - loss: 2.0125
Epoch 2/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 318s 36ms/step - loss: 1.6785
Epoch 3/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 345s 40ms/step - loss: 1.6108
Epoch 4/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 302s 35ms/step - loss: 1.5754
Epoch 5/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 303s 35ms/step - loss: 1.5529
Epoch 6/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 303s 35ms/step - loss: 1.5375
Epoch 7/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 309s 35ms/step - loss: 1.5249
Epoch 8/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 327s 38ms/step - loss: 1.5149


'to be so long the strangence, and so strangers of the strangence, and so strangers of the strangence, and'

In [148]:
m9 = keras.Sequential(
        [
                keras.layers.Input(shape=(None,), dtype="int32"),
                keras.layers.Embedding(input_dim=n_bpe_tokens, output_dim=64),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.Dense(n_bpe_tokens, activation="softmax"),
                ]
        )
m9.compile(loss="sparse_categorical_crossentropy", optimizer="nadam")
m9.summary()

Model: "sequential_35"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_16 (Embedding)        │ (None, None, 64)       │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_41 (GRU)                    │ (None, None, 64)       │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_42 (GRU)                    │ (None, None, 64)       │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_43 (GRU)                    │ (None, None, 64)       │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, None, 512)      │        33,280 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 140,928 (550.50 KB)

 Trainable params: 140,928 (550.50 KB)

 Non-trainable params: 0 (0.00 B)

In [149]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.03, patience=2)
history = m9.fit(
        bpe_ds,
        epochs=10,
        shuffle=False,  # The DataLoader controls shuffling.
        callbacks=[early_stopping],
        verbose=1
        )
m9.save("../models/shakespeare_lstm_9.keras")
tools.generate_text(m9, bpe_tokenizer, "to be", n_next=100, one_hot=False)

Epoch 1/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 657s 75ms/step - loss: 2.3833 
Epoch 2/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 621s 71ms/step - loss: 1.7725
Epoch 3/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 626s 72ms/step - loss: 1.6705
Epoch 4/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 652s 75ms/step - loss: 1.6193
Epoch 5/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 640s 73ms/step - loss: 1.5866 
Epoch 6/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 710s 81ms/step - loss: 1.5646
Epoch 7/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 671s 77ms/step - loss: 1.5473
Epoch 8/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 635s 73ms/step - loss: 1.5339
Epoch 9/10
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 625s 72ms/step - loss: 1.5233 


'to be so speak the speak the speak the speak the speak\nWith the speak the speak the speak the speak the s'

### Tokenization with BOS and EOS tokens

In [ ]:
import re

sentences = re.split(r'(?<=[.!?])\s+', shakespeare_text.strip())
sentences = [s for s in sentences if s.strip()]

print(f"Anzahl Sätze: {len(sentences):,}")

In [ ]:
from tokenizers.processors import TemplateProcessing

BOS = "<|begin_of_text|>"
EOS = "<|end_of_text|>"

# BPE-Tokenizer erstellen
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

# Vokabular trainieren
trainer = trainers.BpeTrainer(
        vocab_size=512,
        special_tokens=["[UNK]", BOS, EOS],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
        )

tokenizer.train_from_iterator([shakespeare_text], trainer=trainer)

# IDs der Special Tokens
bos_id = tokenizer.token_to_id(BOS)
eos_id = tokenizer.token_to_id(EOS)

# BOS und EOS automatisch hinzufügen
tokenizer.post_processor = TemplateProcessing(
        single=f"{BOS} $A {EOS}",
        special_tokens=[
                (BOS, bos_id),
                (EOS, eos_id)
                ]
        )

n_bpe_tokens = tokenizer.get_vocab_size()
print("BPE-Vokabulargröße:", n_bpe_tokens)

# Text tokenisieren
ids = tokenizer.encode(shakespeare_text).ids

print(f"Tokenanzahl: {len(ids):,}")
print("Beispiel:", ids[:30])

# Zurück in Text
print(tokenizer.decode(ids[:30], skip_special_tokens=False))

In [ ]:
# Alle Sätze tokenisieren
token_ids_template = [
        token_id
        for sentence in sentences
        for token_id in tokenizer.encode(sentence).ids
        ]

In [ ]:
token_ids_loop = []

for sentence in sentences:
    ids = tokenizer.encode(
            sentence,
            add_special_tokens=False
            ).ids

    token_ids_loop.append(bos_id)
    token_ids_loop.extend(ids)
    token_ids_loop.append(eos_id)

In [ ]:
print("Identisch:", token_ids_template == token_ids_loop)
print("Tokenanzahl:", len(token_ids_template))

print(
        tokenizer.decode(
                token_ids_template[:100],
                skip_special_tokens=False
                )
        )

In [ ]:
encodings = tokenizer.encode_batch(sentences)

token_ids = [
        token_id
        for encoding in encodings
        for token_id in encoding.ids
        ]